
# Examen de Robótica  
**Alumno:** Camacho Bernabé Roberto Ángel

## Introducción  
En la robótica, el estudio de la cinemática y la dinámica de los manipuladores constituye uno de los pilares fundamentales, ya que permite la comprensión rigurosa de los movimientos y las fuerzas que los originan. Este reporte analiza el robot SCARA, modelando su comportamiento cinemático y dinámico para simulaciones o sistemas de control.  
Se desarrolla el modelo cinemático directo e inverso de postura, velocidades y aceleraciones del efector final, estableciendo una relación matemática entre el espacio articular y cartesiano. También se construye el modelo dinámico mediante el método de Euler-Lagrange.  
El desarrollo fue realizado con MATLAB, y se convierte aquí a Python simbólico usando SymPy.


## 2. Modelado cinemático de la postura

In [1]:

from sympy import zeros, symbols, cos, sin, Matrix, simplify, diag

# Definición simbólica
x_i_j, y_i_j, z_i_j, gi_j, bi_j, ai_j = symbols('x_i_j y_i_j z_i_j gi_j bi_j ai_j')

def Tij(x, y, z, gamma, beta, alpha):
    return Matrix([
        [cos(alpha)*cos(beta), cos(alpha)*sin(beta)*sin(gamma)-sin(alpha)*cos(gamma), cos(alpha)*sin(beta)*cos(gamma)+sin(alpha)*sin(gamma), x],
        [sin(alpha)*cos(beta), sin(alpha)*sin(beta)*sin(gamma)+cos(alpha)*cos(gamma), sin(alpha)*sin(beta)*cos(gamma)-cos(alpha)*sin(gamma), y],
        [-sin(beta),            cos(beta)*sin(gamma),                                cos(beta)*cos(gamma),                                z],
        [0,                    0,                                                    0,                                                    1]
    ])


### 2.1 Modelo cinemático directo de la postura

In [2]:

# Parámetros simbólicos
x_O_1, y_O_1, theta_O_1 = symbols('x_O_1 y_O_1 theta_O_1')
L_1, L_2, L_3 = symbols('L_1 L_2 L_3')
theta_1_2, theta_2_3 = symbols('theta_1_2 theta_2_3')

T_O_1 = Tij(x_O_1, y_O_1, 0, 0, 0, theta_O_1)
T_1_2 = Tij(L_1, 0, 0, 0, 0, theta_1_2)
T_2_3 = Tij(L_2, 0, 0, 0, 0, theta_2_3)
T_3_P = Tij(L_3, 0, 0, 0, 0, 0)

T_O_P = simplify(T_O_1 * T_1_2 * T_2_3 * T_3_P)
T_O_P


Matrix([
[cos(theta_1_2 + theta_2_3 + theta_O_1), -sin(theta_1_2 + theta_2_3 + theta_O_1), 0, L_1*cos(theta_O_1) + L_2*cos(theta_1_2 + theta_O_1) + L_3*cos(theta_1_2 + theta_2_3 + theta_O_1) + x_O_1],
[sin(theta_1_2 + theta_2_3 + theta_O_1),  cos(theta_1_2 + theta_2_3 + theta_O_1), 0, L_1*sin(theta_O_1) + L_2*sin(theta_1_2 + theta_O_1) + L_3*sin(theta_1_2 + theta_2_3 + theta_O_1) + y_O_1],
[                                     0,                                       0, 1,                                                                                                        0],
[                                     0,                                       0, 0,                                                                                                        1]])

### 2.2 Modelo cinemático inverso de la postura

In [3]:

# Vector de postura del efector
xi_O_P = Matrix([T_O_P[0, 3], T_O_P[1, 3], theta_O_1 + theta_1_2 + theta_2_3])
xi_O_P


Matrix([
[L_1*cos(theta_O_1) + L_2*cos(theta_1_2 + theta_O_1) + L_3*cos(theta_1_2 + theta_2_3 + theta_O_1) + x_O_1],
[L_1*sin(theta_O_1) + L_2*sin(theta_1_2 + theta_O_1) + L_3*sin(theta_1_2 + theta_2_3 + theta_O_1) + y_O_1],
[                                                                       theta_1_2 + theta_2_3 + theta_O_1]])

## 3. Modelo cinemático de las velocidades

In [4]:
# Variables simbólicas
theta_dot_O_1, theta_dot_1_2, theta_dot_2_3 = symbols('theta_dot_O_1 theta_dot_1_2 theta_dot_2_3')

# Jacobiano usando método de Matrix
J_theta = xi_O_P.jacobian([theta_O_1, theta_1_2, theta_2_3])
J_theta = simplify(J_theta)

# Vector de velocidades articulares
theta_dot = Matrix([theta_dot_O_1, theta_dot_1_2, theta_dot_2_3])

# Velocidades del efector final
xi_dot_O_P = simplify(J_theta * theta_dot)
xi_dot_O_P


Matrix([
[-L_1*theta_dot_O_1*sin(theta_O_1) - L_2*theta_dot_1_2*sin(theta_1_2 + theta_O_1) - L_2*theta_dot_O_1*sin(theta_1_2 + theta_O_1) - L_3*theta_dot_1_2*sin(theta_1_2 + theta_2_3 + theta_O_1) - L_3*theta_dot_2_3*sin(theta_1_2 + theta_2_3 + theta_O_1) - L_3*theta_dot_O_1*sin(theta_1_2 + theta_2_3 + theta_O_1)],
[                                       L_3*theta_dot_2_3*cos(theta_1_2 + theta_2_3 + theta_O_1) + theta_dot_1_2*(L_2*cos(theta_1_2 + theta_O_1) + L_3*cos(theta_1_2 + theta_2_3 + theta_O_1)) + theta_dot_O_1*(L_1*cos(theta_O_1) + L_2*cos(theta_1_2 + theta_O_1) + L_3*cos(theta_1_2 + theta_2_3 + theta_O_1))],
[                                                                                                                                                                                                                                                                   theta_dot_1_2 + theta_dot_2_3 + theta_dot_O_1]])

### 3.2 Modelo cinemático inverso de las velocidades

In [5]:

# Inversa del Jacobiano
J_theta_inv = simplify(J_theta.inv())

# Velocidades del efector como entrada
x_dot, y_dot, phi_dot = symbols('x_dot y_dot phi_dot')
xi_dot = Matrix([x_dot, y_dot, phi_dot])

# Cálculo de velocidades articulares a partir de velocidades del efector
theta_dot_from_xidot = simplify(J_theta_inv * xi_dot)
theta_dot_from_xidot


Matrix([
[                                                                                                                    (L_3*phi_dot*sin(theta_2_3) + x_dot*cos(theta_1_2 + theta_O_1) + y_dot*sin(theta_1_2 + theta_O_1))/(L_1*sin(theta_1_2))],
[-(L_1*L_3*phi_dot*sin(theta_1_2 + theta_2_3) + L_1*x_dot*cos(theta_O_1) + L_1*y_dot*sin(theta_O_1) + L_2*L_3*phi_dot*sin(theta_2_3) + L_2*x_dot*cos(theta_1_2 + theta_O_1) + L_2*y_dot*sin(theta_1_2 + theta_O_1))/(L_1*L_2*sin(theta_1_2))],
[                                                                     (L_2*phi_dot + L_3*phi_dot*sin(theta_2_3)/tan(theta_1_2) + L_3*phi_dot*cos(theta_2_3) + x_dot*cos(theta_O_1)/sin(theta_1_2) + y_dot*sin(theta_O_1)/sin(theta_1_2))/L_2]])

## 4. Modelo cinemático de las aceleraciones

In [6]:

from sympy import zeros

# Variables simbólicas adicionales para aceleraciones
theta_ddot_O_1, theta_ddot_1_2, theta_ddot_2_3 = symbols('theta_ddot_O_1 theta_ddot_1_2 theta_ddot_2_3')

# Vectores de velocidades y aceleraciones articulares
q_theta = Matrix([theta_O_1, theta_1_2, theta_2_3])
q_dot = Matrix([theta_dot_O_1, theta_dot_1_2, theta_dot_2_3])
q_ddot = Matrix([theta_ddot_O_1, theta_ddot_1_2, theta_ddot_2_3])

# Derivada temporal del Jacobiano
J_dot = zeros(*J_theta.shape)
for i in range(3):
    J_dot += J_theta.diff(q_theta[i]) * q_dot[i]

# Aceleración del efector final
xi_ddot_O_P = simplify(J_theta * q_ddot + J_dot * q_dot)
xi_ddot_O_P


Matrix([
[-L_1*theta_ddot_O_1*sin(theta_O_1) - L_1*theta_dot_O_1**2*cos(theta_O_1) - L_2*theta_ddot_1_2*sin(theta_1_2 + theta_O_1) - L_2*theta_ddot_O_1*sin(theta_1_2 + theta_O_1) - L_2*theta_dot_1_2**2*cos(theta_1_2 + theta_O_1) - 2*L_2*theta_dot_1_2*theta_dot_O_1*cos(theta_1_2 + theta_O_1) - L_2*theta_dot_O_1**2*cos(theta_1_2 + theta_O_1) - L_3*theta_ddot_1_2*sin(theta_1_2 + theta_2_3 + theta_O_1) - L_3*theta_ddot_2_3*sin(theta_1_2 + theta_2_3 + theta_O_1) - L_3*theta_ddot_O_1*sin(theta_1_2 + theta_2_3 + theta_O_1) - L_3*theta_dot_1_2**2*cos(theta_1_2 + theta_2_3 + theta_O_1) - 2*L_3*theta_dot_1_2*theta_dot_2_3*cos(theta_1_2 + theta_2_3 + theta_O_1) - 2*L_3*theta_dot_1_2*theta_dot_O_1*cos(theta_1_2 + theta_2_3 + theta_O_1) - L_3*theta_dot_2_3**2*cos(theta_1_2 + theta_2_3 + theta_O_1) - 2*L_3*theta_dot_2_3*theta_dot_O_1*cos(theta_1_2 + theta_2_3 + theta_O_1) - L_3*theta_dot_O_1**2*cos(theta_1_2 + theta_2_3 + theta_O_1)],
[ L_1*theta_ddot_O_1*cos(theta_O_1) - L_1*theta_dot_O_1**2*sin(th

In [7]:

# 4.3 Modelo cinemático inverso de las aceleraciones
# Resolución de q̈ a partir de xï (aceleraciones del efector)
# q̈ = J^{-1} * (xï - J̇ * q̇)
q_ddot_from_xiddot = J_theta.LUsolve(xi_ddot_O_P - J_dot * q_dot)
q_ddot_from_xiddot


Matrix([
[theta_ddot_1_2 + theta_ddot_2_3 + theta_ddot_O_1 - (-L_1*theta_ddot_O_1*sin(theta_O_1) - L_1*theta_dot_O_1**2*cos(theta_O_1) - L_2*theta_ddot_1_2*sin(theta_1_2 + theta_O_1) - L_2*theta_ddot_O_1*sin(theta_1_2 + theta_O_1) - L_2*theta_dot_1_2**2*cos(theta_1_2 + theta_O_1) - 2*L_2*theta_dot_1_2*theta_dot_O_1*cos(theta_1_2 + theta_O_1) - L_2*theta_dot_O_1**2*cos(theta_1_2 + theta_O_1) - L_3*theta_ddot_1_2*sin(theta_1_2 + theta_2_3 + theta_O_1) - L_3*theta_ddot_2_3*sin(theta_1_2 + theta_2_3 + theta_O_1) - L_3*theta_ddot_O_1*sin(theta_1_2 + theta_2_3 + theta_O_1) - L_3*theta_dot_1_2**2*cos(theta_1_2 + theta_2_3 + theta_O_1) - 2*L_3*theta_dot_1_2*theta_dot_2_3*cos(theta_1_2 + theta_2_3 + theta_O_1) - 2*L_3*theta_dot_1_2*theta_dot_O_1*cos(theta_1_2 + theta_2_3 + theta_O_1) - L_3*theta_dot_2_3**2*cos(theta_1_2 + theta_2_3 + theta_O_1) - 2*L_3*theta_dot_2_3*theta_dot_O_1*cos(theta_1_2 + theta_2_3 + theta_O_1) - L_3*theta_dot_O_1**2*cos(theta_1_2 + theta_2_3 + theta_O_1) - theta_dot_1_2

## 5. Modelo dinámico por ecuaciones de Euler-Lagrange

In [8]:

# Variables simbólicas para energía y masas
x_1_C1, x_2_C2, x_3_C3 = symbols('x_1_C1 x_2_C2 x_3_C3')
m_1, m_2, m_3 = symbols('m_1 m_2 m_3')
g = symbols('g')

# Inercias
I_xx1, I_yy1, I_zz1 = symbols('I_xx1 I_yy1 I_zz1')
I_xx2, I_yy2, I_zz2 = symbols('I_xx2 I_yy2 I_zz2')
I_xx3, I_yy3, I_zz3 = symbols('I_xx3 I_yy3 I_zz3')

I_C1 = diag(I_xx1, I_yy1, I_zz1)
I_C2 = diag(I_xx2, I_yy2, I_zz2)
I_C3 = diag(I_xx3, I_yy3, I_zz3)

# Vector gravedad
g_v = Matrix([0, -g, 0])


In [9]:

# Transformaciones de los centros de masa
T_1_C1 = Tij(x_1_C1, 0, 0, 0, 0, 0)
T_2_C2 = Tij(x_2_C2, 0, 0, 0, 0, 0)
T_3_C3 = Tij(x_3_C3, 0, 0, 0, 0, 0)

T_O_C1 = simplify(T_O_1 * T_1_C1)
T_O_C2 = simplify(T_O_1 * T_1_2 * T_2_C2)
T_O_C3 = simplify(T_O_1 * T_1_2 * T_2_3 * T_3_C3)

# Vectores de posición
p_O_C1 = T_O_C1[:3, 3]
p_O_C2 = T_O_C2[:3, 3]
p_O_C3 = T_O_C3[:3, 3]


In [10]:

# Velocidades lineales de los centros de masa
v_O_C1 = zeros(3, 1)
v_O_C2 = zeros(3, 1)
v_O_C3 = zeros(3, 1)

for i in range(3):
    v_O_C1 += p_O_C1.diff(q_theta[i]) * q_dot[i]
    v_O_C2 += p_O_C2.diff(q_theta[i]) * q_dot[i]
    v_O_C3 += p_O_C3.diff(q_theta[i]) * q_dot[i]

# Simplificar resultados
v_O_C1 = simplify(v_O_C1)
v_O_C2 = simplify(v_O_C2)
v_O_C3 = simplify(v_O_C3)


In [11]:

# Matrices de rotación
R_O_1 = T_O_1[:3, :3]
R_1_2 = T_1_2[:3, :3]
R_2_3 = T_2_3[:3, :3]

# Propagación de velocidades angulares
n = Matrix([0, 0, 1])
omega_0 = Matrix([0, 0, 0])

omega_1 = R_O_1.T * omega_0 + n * theta_dot_O_1
omega_2 = R_1_2.T * omega_1 + n * theta_dot_1_2
omega_3 = R_2_3.T * omega_2 + n * theta_dot_2_3


In [12]:

# Energía cinética
k_1 = simplify((m_1/2) * v_O_C1.dot(v_O_C1) + (omega_1.dot(I_C1 * omega_1))/2)
k_2 = simplify((m_2/2) * v_O_C2.dot(v_O_C2) + (omega_2.dot(I_C2 * omega_2))/2)
k_3 = simplify((m_3/2) * v_O_C3.dot(v_O_C3) + (omega_3.dot(I_C3 * omega_3))/2)


In [13]:

# Energía potencial
u_1 = -m_1 * p_O_C1.dot(g_v)
u_2 = -m_2 * p_O_C2.dot(g_v)
u_3 = -m_3 * p_O_C3.dot(g_v)


In [14]:

# Lagrangiano total
La = simplify((k_1 + k_2 + k_3) - (u_1 + u_2 + u_3))
La


I_zz2*(theta_dot_1_2 + theta_dot_O_1)**2/2 + I_zz3*(theta_dot_1_2 + theta_dot_2_3 + theta_dot_O_1)**2/2 - g*m_1*(x_1_C1*sin(theta_O_1) + y_O_1) - g*m_2*(L_1*sin(theta_O_1) + x_2_C2*sin(theta_1_2 + theta_O_1) + y_O_1) - g*m_3*(L_1*sin(theta_O_1) + L_2*sin(theta_1_2 + theta_O_1) + x_3_C3*sin(theta_1_2 + theta_2_3 + theta_O_1) + y_O_1) + m_2*(L_1**2*theta_dot_O_1**2 + 2*L_1*theta_dot_1_2*theta_dot_O_1*x_2_C2*cos(theta_1_2) + 2*L_1*theta_dot_O_1**2*x_2_C2*cos(theta_1_2) + theta_dot_1_2**2*x_2_C2**2 + 2*theta_dot_1_2*theta_dot_O_1*x_2_C2**2 + theta_dot_O_1**2*x_2_C2**2)/2 + m_3*(L_1**2*theta_dot_O_1**2 + 2*L_1*L_2*theta_dot_1_2*theta_dot_O_1*cos(theta_1_2) + 2*L_1*L_2*theta_dot_O_1**2*cos(theta_1_2) + 2*L_1*theta_dot_1_2*theta_dot_O_1*x_3_C3*cos(theta_1_2 + theta_2_3) + 2*L_1*theta_dot_2_3*theta_dot_O_1*x_3_C3*cos(theta_1_2 + theta_2_3) + 2*L_1*theta_dot_O_1**2*x_3_C3*cos(theta_1_2 + theta_2_3) + L_2**2*theta_dot_1_2**2 + 2*L_2**2*theta_dot_1_2*theta_dot_O_1 + L_2**2*theta_dot_O_1**2 + 2*L_

In [15]:
# Cálculo de los torques con Ecuaciones de Euler-Lagrange
tau_list = []
for i in range(3):
    dLa_dqdot = La.diff(q_dot[i])
    d_dt = sum([dLa_dqdot.diff(q_theta[j]) * q_dot[j] + dLa_dqdot.diff(q_dot[j]) * q_ddot[j] for j in range(3)])
    tau_i = simplify(d_dt - La.diff(q_theta[i]))
    tau_list.append(tau_i)
tao = Matrix(tau_list)
tao


Matrix([
[-L_1*theta_dot_1_2*(m_2*x_2_C2*(theta_dot_1_2 + 2*theta_dot_O_1)*sin(theta_1_2) + m_3*(L_2*theta_dot_1_2*sin(theta_1_2) + 2*L_2*theta_dot_O_1*sin(theta_1_2) + theta_dot_1_2*x_3_C3*sin(theta_1_2 + theta_2_3) + theta_dot_2_3*x_3_C3*sin(theta_1_2 + theta_2_3) + 2*theta_dot_O_1*x_3_C3*sin(theta_1_2 + theta_2_3))) + g*m_1*x_1_C1*cos(theta_O_1) + g*m_2*(L_1*cos(theta_O_1) + x_2_C2*cos(theta_1_2 + theta_O_1)) + g*m_3*(L_1*cos(theta_O_1) + L_2*cos(theta_1_2 + theta_O_1) + x_3_C3*cos(theta_1_2 + theta_2_3 + theta_O_1)) - m_3*theta_dot_2_3*x_3_C3*(L_1*theta_dot_1_2*sin(theta_1_2 + theta_2_3) + L_1*theta_dot_2_3*sin(theta_1_2 + theta_2_3) + 2*L_1*theta_dot_O_1*sin(theta_1_2 + theta_2_3) + 2*L_2*theta_dot_1_2*sin(theta_2_3) + L_2*theta_dot_2_3*sin(theta_2_3) + 2*L_2*theta_dot_O_1*sin(theta_2_3)) + theta_ddot_1_2*(I_zz2 + I_zz3 + m_2*x_2_C2*(L_1*cos(theta_1_2) + x_2_C2) + m_3*(L_1*L_2*cos(theta_1_2) + L_1*x_3_C3*cos(theta_1_2 + theta_2_3) + L_2**2 + 2*L_2*x_3_C3*cos(theta_2_3) + x_3_C3**2

In [16]:

# Identificación M, V, G en la ecuación: tau = M*q_ddot + V + G
M1 = tao.subs({theta_ddot_O_1:1, theta_ddot_1_2:0, theta_ddot_2_3:0, theta_dot_O_1:0, theta_dot_1_2:0, theta_dot_2_3:0, g:0})
M2 = tao.subs({theta_ddot_O_1:0, theta_ddot_1_2:1, theta_ddot_2_3:0, theta_dot_O_1:0, theta_dot_1_2:0, theta_dot_2_3:0, g:0})
M3 = tao.subs({theta_ddot_O_1:0, theta_ddot_1_2:0, theta_ddot_2_3:1, theta_dot_O_1:0, theta_dot_1_2:0, theta_dot_2_3:0, g:0})
M_theta = simplify(Matrix.hstack(M1, M2, M3))

V_theta = tao.subs({theta_ddot_O_1:0, theta_ddot_1_2:0, theta_ddot_2_3:0, g:0})
G_theta = tao.subs({theta_ddot_O_1:0, theta_ddot_1_2:0, theta_ddot_2_3:0, theta_dot_O_1:0, theta_dot_1_2:0, theta_dot_2_3:0})

tau_inverse = simplify(M_theta * q_ddot + V_theta + G_theta)
tau_inverse


Matrix([
[-L_1*theta_dot_1_2*(m_2*x_2_C2*(theta_dot_1_2 + 2*theta_dot_O_1)*sin(theta_1_2) + m_3*(L_2*theta_dot_1_2*sin(theta_1_2) + 2*L_2*theta_dot_O_1*sin(theta_1_2) + theta_dot_1_2*x_3_C3*sin(theta_1_2 + theta_2_3) + theta_dot_2_3*x_3_C3*sin(theta_1_2 + theta_2_3) + 2*theta_dot_O_1*x_3_C3*sin(theta_1_2 + theta_2_3))) + g*m_1*x_1_C1*cos(theta_O_1) + g*m_2*(L_1*cos(theta_O_1) + x_2_C2*cos(theta_1_2 + theta_O_1)) + g*m_3*(L_1*cos(theta_O_1) + L_2*cos(theta_1_2 + theta_O_1) + x_3_C3*cos(theta_1_2 + theta_2_3 + theta_O_1)) - m_3*theta_dot_2_3*x_3_C3*(L_1*theta_dot_1_2*sin(theta_1_2 + theta_2_3) + L_1*theta_dot_2_3*sin(theta_1_2 + theta_2_3) + 2*L_1*theta_dot_O_1*sin(theta_1_2 + theta_2_3) + 2*L_2*theta_dot_1_2*sin(theta_2_3) + L_2*theta_dot_2_3*sin(theta_2_3) + 2*L_2*theta_dot_O_1*sin(theta_2_3)) + theta_ddot_1_2*(I_zz2 + I_zz3 + m_2*x_2_C2*(L_1*cos(theta_1_2) + x_2_C2) + m_3*(L_1*L_2*cos(theta_1_2) + L_1*x_3_C3*cos(theta_1_2 + theta_2_3) + L_2**2 + 2*L_2*x_3_C3*cos(theta_2_3) + x_3_C3**2

In [ ]:

# 6. Modelo dinámico directo
# q̈ = M^{-1} * (τ - V - G)
q_ddot = simplify(M_theta.inv() * (tao - V_theta - G_theta))
q_ddot


## Conclusión


Se reforzaron los conocimientos adquiridos en la clase de Robótica mediante el modelado cinemático y dinámico del robot SCARA.  
Se desarrollaron modelos directos e inversos de postura, velocidades y aceleraciones, estableciendo relaciones matemáticas clave entre el espacio articular y cartesiano.  
También se modeló la dinámica con el método de Euler-Lagrange, obteniendo la matriz de inercia, efectos centrífugos y gravitacionales.  
Esto proporciona una base sólida para la implementación en simulaciones y sistemas de control en tiempo real.
